In [ ]:
from google.colab import files
uploaded = files.upload()   # choose your CSV file here


In [ ]:
import pandas as pd

df = pd.read_csv("source_credibility_dataset_200.csv")  # use the exact file name shown
df.head()


In [ ]:
import os

if 'source_credibility_dataset_200.csv' in os.listdir('.'):
    print('File source_credibility_dataset_200.csv uploaded successfully!')
else:
    print('File source_credibility_dataset_200.csv not found after upload. Please ensure you uploaded the correct file name.')

After successfully uploading the file, you can re-run the cell `A34UrHV5csxo` to load the dataset.

Preprocess the data

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Create extra features
df["reliability_ratio"] = df["past_real"] / (df["past_real"] + df["past_fake"] + 1e-6)
df["fake_ratio"] = df["past_fake"] / (df["past_real"] + df["past_fake"] + 1e-6)

# 2. Encode language (Tamil/Sinhala/English) into numbers
lang_le = LabelEncoder()
df["language_encoded"] = lang_le.fit_transform(df["language"])

# 3. Choose features for the model
feature_cols = [
    "past_fake",
    "past_real",
    "domain_age_years",
    "followers",
    "reliability_ratio",
    "fake_ratio",
    "language_encoded"
]

X = df[feature_cols].values
y_labels = df["credibility_label"].values

# 4. Encode labels (High/Medium/Low -> 2/1/0 etc.)
label_le = LabelEncoder()
y = label_le.fit_transform(y_labels)

print("Label mapping:", dict(zip(label_le.classes_, label_le.transform(label_le.classes_))))

# 5. Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 6. Scale numeric values (very important for ML)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Preprocessing Done!")


Train first model (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Create the model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

# Train the model
rf.fit(X_train_scaled, y_train)

# Predict on test data
y_pred = rf.predict(X_test_scaled)

print("✅ Model trained!")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_le.classes_))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Save trained model (for backend / later use)

In [ ]:
import joblib

# Save models and encoders
joblib.dump(rf, "source_credibility_rf_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(label_le, "label_encoder.pkl")
joblib.dump(lang_le, "language_encoder.pkl")

from google.colab import files
files.download("source_credibility_rf_model.pkl")
files.download("feature_scaler.pkl")
files.download("label_encoder.pkl")
files.download("language_encoder.pkl")


Test prediction on a new sample

In [ ]:
def predict_source(past_fake, past_real, domain_age_years, followers, language_str):
    import numpy as np

    # compute extra features
    reliability_ratio = past_real / (past_real + past_fake + 1e-6)
    fake_ratio = past_fake / (past_real + past_fake + 1e-6)

    # encode language
    lang_code = lang_le.transform([language_str])[0]

    # feature vector
    x = np.array([[past_fake, past_real, domain_age_years,
                   followers, reliability_ratio, fake_ratio, lang_code]])

    # scale
    x_scaled = scaler.transform(x)

    # predict
    pred = rf.predict(x_scaled)[0]
    label = label_le.inverse_transform([pred])[0]
    return label

predict_source(
    past_fake=3,
    past_real=500,
    domain_age_years=10,
    followers=800000,
    language_str="Tamil"   # or "Sinhala" / "English"
)


Prediction with probabilities

In [ ]:
def predict_with_prob(past_fake, past_real, domain_age_years, followers, language_str):
    # compute extra features
    reliability_ratio = past_real / (past_real + past_fake + 1e-6)
    fake_ratio = past_fake / (past_real + past_fake + 1e-6)

    # encode language
    lang_code = lang_le.transform([language_str])[0]

    # feature vector (same order as training)
    x = np.array([[past_fake,
                   past_real,
                   domain_age_years,
                   followers,
                   reliability_ratio,
                   fake_ratio,
                   lang_code]])

    # scale
    x_scaled = scaler.transform(x)

    # predict probabilities
    proba = rf.predict_proba(x_scaled)[0]
    pred_idx = int(np.argmax(proba))
    label = label_le.inverse_transform([pred_idx])[0]

    # map probabilities to class names
    proba_dict = {cls: float(p) for cls, p in zip(label_le.classes_, proba)}

    return label, proba_dict

# Example:
predict_with_prob(
    past_fake=3,
    past_real=500,
    domain_age_years=10,
    followers=800000,
    language_str="Tamil"
)


Feature importance

In [ ]:
# Feature importance analysis
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]  # sort high → low

print("Feature importances (highest to lowest):\n")
for idx in indices:
    print(f"{feature_cols[idx]:25s}  {importances[idx]:.3f}")


Bar Chart

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)),
           [feature_cols[i] for i in indices],
           rotation=45, ha="right")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()


Cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf,
    X_train_scaled,
    y_train,
    cv=5,
    scoring="f1_macro"
)

print("5-fold CV macro F1 scores:", cv_scores)
print("Mean F1:", cv_scores.mean(), "±", cv_scores.std())


Save metadata + all artifacts together

In [ ]:
import json
import sklearn
import sys

# Label mapping (class → code)
# Convert numpy.int64 to standard int for JSON serialization
label_mapping = {cls: int(code) for cls, code in zip(label_le.classes_, label_le.transform(label_le.classes_))}

metadata = {
    "feature_columns": feature_cols,
    "label_mapping": label_mapping,
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version,
}

with open("source_credibility_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata:")
print(json.dumps(metadata, indent=2))

In [ ]:
from google.colab import files

files_to_download = [
    "source_credibility_rf_model.pkl",
    "feature_scaler.pkl",
    "label_encoder.pkl",
    "language_encoder.pkl",
    "source_credibility_metadata.json",
]

for fname in files_to_download:
    files.download(fname)
